#### Define notebook Parameters

Pass through th pl_worker

In [1]:
# Framework-compatible parameters with manual fallbacks.
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load taxonomy to silver"
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False

# Source config
# source_tables: first table is the main source table
source_settings = json.dumps({
    "source_name": "taxonomy",
    "table_config": [
        {
            "schema_name": "external_curated_taxonomy",
            "source_tables": ["curated_band_level"],
            "target_table": "silver_taxonomy_curated_band_level",
            "primary_keys": ["band_level_code"],
            "is_incremental": 0
        },
        {
            "schema_name": "external_curated_taxonomy",
            "source_tables": ["curated_bill_type"],
            "target_table": "silver_taxonomy_curated_bill_type",
            "primary_keys": ["bill_type_code"],
            "is_incremental": 0
        }
    ]
})

# Target config
target_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "schema_name": "silver_taxonomy",
    "load_strategy": "merge-scd1",
    "location_root": "Files/silver/taxonomy"
})

StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 3, Finished, Available, Finished, False)

#### Define functions and logging

Used for import functions

In [2]:
%run nb_transformation_shared_functions

StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 8, Finished, Available, Finished, True)

In [3]:
# Set up standard framework logging
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 9, Finished, Available, Finished, False)

#### Define variables or parameters

Can manually run or be through worker

In [4]:
# Parse notebook settings
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings

# Source configuration
source_name = source_settings.get("source_name")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse_name = target_settings.get("lakehouse_name")
target_schema = target_settings.get("schema_name")
target_load_strategy = target_settings.get("load_strategy", "merge-scd1")
location_root = target_settings.get("location_root")

# Validation output
print(source_name)
print(target_schema)
display(table_config)

StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 10, Finished, Available, Finished, False)

taxonomy
silver_taxonomy
2026-07-01 20:41:36,300 UTC - INFO - Calling getAccessToken from PyTridentTokenLibrary (trident_token_library_wrapper)


SynapseWidget(Synapse.DataFrame, 2476593d-b75a-4c80-9039-da72db0907b1)

#### Ingest source table

Read and dedup clean source

In [5]:
# Find every unique source table needed by the target mappings
source_tables_to_read = {}

for cfg in table_config:
    source_schema = cfg.get("schema_name")
    source_tables = cfg.get("source_tables", [])

    if not source_tables:
        raise ValueError(f"Missing source_tables for {cfg.get('target_table')}")

    for source_table in source_tables:
        source_view = f"src_{to_snake_case(source_table)}"

        source_tables_to_read[source_view] = {
            "schema_name": source_schema,
            "source_table": source_table
        }
        
# Read shortcut tables and create src_* temp views
for source_view, source_cfg in source_tables_to_read.items():
    source_schema = source_cfg.get("schema_name")
    source_table = source_cfg.get("source_table")
    full_source_name = f"{source_schema}.{source_table}"

    # Read source table from lakehouse shortcut
    df_source = spark.table(full_source_name)

    # Drop exact duplicates only
    df_source = remove_exact_duplicates(df_source)

    # Create source temp view
    df_source.createOrReplaceTempView(source_view)

    # Preview source data
    print(f"Source preview for {source_view} ({full_source_name})")
    display(df_source.limit(3))

StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 11, Finished, Available, Finished, False)

Source preview for src_curated_band_level (external_curated_taxonomy.curated_band_level)


SynapseWidget(Synapse.DataFrame, 386a8b58-ce8d-48f9-8841-9681423aed7a)

Source preview for src_curated_bill_type (external_curated_taxonomy.curated_bill_type)


SynapseWidget(Synapse.DataFrame, 5eab68b0-5a25-4e26-9277-da43ebe23e2c)

#### Transform source table

Applies mapping and transformation as needed

In [7]:
# Map source views into target shape
for cfg in table_config:
    # Get source tables 
    source_tables = cfg.get("source_tables", [])
    main_source_table = source_tables[0]
    main_source_view = f"src_{to_snake_case(main_source_table)}"

    # Get target tables
    target_table = cfg.get("target_table")
    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    # Build map views
    map_view = f"map_{target_entity}"

    # Map Band Level
    if target_entity == "curated_band_level":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT
            band_level_key,
            band_level_code,
            band_level_description,
            band_group_code,
            band_group_description,
            is_active
        FROM {main_source_view}
        """)

    # Map Bill Type
    elif target_entity == "curated_bill_type":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT
            bill_type_key,
            bill_type_code,
            bill_type_description,
            is_active
        FROM {main_source_view}
        """)

    else:
        raise ValueError(f"No mapping defined for {target_table}")

    # Preview mapped data
    print(f"Mapped preview for {map_view}")
    display(spark.table(map_view).limit(3))

StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 13, Finished, Available, Finished, False)

Mapped preview for map_curated_band_level


SynapseWidget(Synapse.DataFrame, ef122905-9a77-4886-9d06-9a9d311fe858)

Mapped preview for map_curated_bill_type


SynapseWidget(Synapse.DataFrame, 95592c6b-63f1-4e30-a3ec-392ba94f4410)

#### Add metadata to source table

Applies metadata and _hashed_pk

In [8]:
# Add metadata and _hashed_pk to mapped views
# This creates final tgt_* views used by validation and merge
for cfg in table_config:
    source_schema = cfg.get("schema_name")
    source_tables = cfg.get("source_tables", [])
    target_table = cfg.get("target_table")
    primary_keys = cfg.get("primary_keys", [])

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    map_view = f"map_{target_entity}"
    target_view = f"tgt_{target_entity}"
    source_file = ", ".join([f"{source_schema}.{source_table}" for source_table in source_tables])

    # Read mapped temp view
    df_target = spark.table(map_view)

    # Build hashed primary key
    pk_expr_cols = [to_snake_case(col) for col in primary_keys]

    # Create _hashed_pk from primary key
    df_target = df_target.withColumn(
        "_hashed_pk",
        F.md5(F.concat_ws("|", *[F.col(col).cast("string") for col in pk_expr_cols]))
    )

    # Get non-business columns to build hashed row
    row_hash_cols = [
        col for col in df_target.columns
        if col not in [df_target.columns[0], "_hashed_pk"] + pk_expr_cols
    ]

    # Create _hashed_row from non-business columns
    df_target = df_target.withColumn(
        "_hashed_row",
        F.md5(F.concat_ws("|", *[F.coalesce(F.col(col).cast("string"), F.lit("<NULL>")) for col in row_hash_cols]))
    )

    # Add framework metadata
    df_target = add_metadata(
        df=df_target,
        ingest_date=str(date.today()),
        file_path=source_file,
        schema_name=source_name,
        table_name=target_table,
        lineage_id=lineage_id
    )

    # Create final target temp view
    df_target.createOrReplaceTempView(target_view)

    # Preview final data
    print(f"Final preview for {target_view}")
    display(df_target.limit(3))

StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 14, Finished, Available, Finished, False)

Final preview for tgt_curated_band_level


SynapseWidget(Synapse.DataFrame, 8c4b171d-3798-440b-a600-a72a0a71a8fc)

Final preview for tgt_curated_bill_type


SynapseWidget(Synapse.DataFrame, 70931b2a-91d7-48b2-a1c3-2917125129fc)

#### Check duplicates

Return duplicates before merge 

In [9]:
# Check duplicate hashed keys before merge
for cfg in table_config:
    target_table = cfg.get("target_table")

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    target_view = f"tgt_{target_entity}"

    # Validate duplicate primary keys
    print(f"Checking duplicates for {target_view}")
    validate_duplicates(spark.table(target_view), "_hashed_pk", 10)

    # to do later
    # DO NOT TRIGGER ERROR IF DUPLICATS, CONTINUE BUT SAVE RECONDS ON dupliate tABLE


StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 15, Finished, Available, Finished, False)

Checking duplicates for tgt_curated_band_level
Total duplicate _hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, 134ee103-3118-4af3-8a26-4cb2dc0f8600)

Checking duplicates for tgt_curated_bill_type
Total duplicate _hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, a14662cb-3bd0-4c08-af2c-94fc3e9050fc)

#### Merge to target

Merge to target table based on _hashed_key

In [10]:
# Merge final temp views into existing target silver tables
load_results = []

for cfg in table_config:
    target_table = cfg.get("target_table")

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    target_view = f"tgt_{target_entity}"
    target_table_path = f"{location_root}/{target_table}"
    full_target_name = f"{target_schema}.{target_table}"

    # Run target load
    df_target = spark.table(target_view)
    result = load_to_target(df_target, target_table_path, target_load_strategy)
    load_results.append(result)

    # Print metrics returned by load_to_target
    print(full_target_name)
    print(f"rows_inserted: {result['rows_inserted']}")
    print(f"rows_updated: {result['rows_updated']}")
    print(f"rows_read: {result['rows_read']}")
    print(f"rows_copied: {result['rows_copied']}")

# Return total metrics to pipeline
result = {
    "rows_inserted": sum(item["rows_inserted"] for item in load_results),
    "rows_updated": sum(item["rows_updated"] for item in load_results),
    "rows_read": sum(item["rows_read"] for item in load_results),
    "rows_copied": sum(item["rows_copied"] for item in load_results)
}

StatementMeta(, 6ef39d37-0f8b-48de-b8bb-d931e334a7c5, 16, Finished, Available, Finished, False)

2026-07-01 20:48:02,053 UTC - INFO - Running merge-scd1 into Files/silver/taxonomy/silver_taxonomy_curated_band_level (LoadTransactionalToBase)
silver_taxonomy.silver_taxonomy_curated_band_level
rows_inserted: 65
rows_updated: 0
rows_read: 65
rows_copied: 0
2026-07-01 20:48:13,771 UTC - INFO - Running merge-scd1 into Files/silver/taxonomy/silver_taxonomy_curated_bill_type (LoadTransactionalToBase)
silver_taxonomy.silver_taxonomy_curated_bill_type
rows_inserted: 79
rows_updated: 0
rows_read: 79
rows_copied: 0
